# الفصل 2: الأسس الرياضية للتعلم العميق

> كل عملية داخل المحوّلات — من الضرب المصفوفي في Attention إلى softmax في طبقة الإخراج إلى دالة الخسارة — هي في جوهرها عملية رياضية محددة.

- **الجزء:** 1 — الأساسيات
- **الفصل:** 2
- **المستوى:** مبتدئ
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 1: إعداد البيئة والبنية التحتية للتطوير
- **الوسوم:** جبر خطي، احتمالات، Softmax، Cross-Entropy

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch02-mathematics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch02-mathematics.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch02-mathematics.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> كل عملية داخل المحوّلات — من الضرب المصفوفي في Attention إلى softmax في طبقة الإخراج إلى دالة الخسارة — هي في جوهرها عملية رياضية محددة.

> **بيانات الفصل**
>
> - **الجزء:** 1 — الأساسيات
> - **المستوى:** مبتدئ
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 1: إعداد البيئة والبنية التحتية للتطوير


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم العلاقة الرياضية الكاملة بين الدوال الثلاث
- فهم الصيغ المرجعية
- فهم جبر خطي


## 1. لماذا يهمّك هذا الموضوع؟

كل عملية داخل المحوّلات — من الضرب المصفوفي في Attention إلى softmax في طبقة الإخراج إلى دالة الخسارة — هي في جوهرها عملية رياضية محددة. من لا يفهم هذه الأسس بعمق سيجد نفسه يتعامل مع المحوّلات كـ"سحر أسود"، عاجزًا عن تشخيص المشاكل (مثل عدم استقرار التدريب العددي) أو ابتكار تحسينات جديدة.


## 2. الخريطة الذهنية


```text
الأسس الرياضية الثلاث لكل نموذج لغوي:

1. الجبر الخطي          → تمثيل البيانات (متجهات، مصفوفات) والعمليات عليها
2. نظرية الاحتمالات     → النموذج اللغوي هو في جوهره توزيع احتمالي P(token|context)
3. نظرية المعلومات      → قياس "الفاقد" بين التوزيع المُتوقَّع والحقيقي (Cross-Entropy)
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس — الجبر الخطي

كل كلمة تُمثَّل كمتجه (نقطة في فضاء متعدد الأبعاد). "التعلّم" هو عملية إيجاد تحويلات خطية (مصفوفات) تُحرّك هذه النقاط في الفضاء بحيث تصبح الكلمات المتشابهة معنويًا قريبة من بعضها، والعمليات الحسابية (كالجمع) ذات معنى (`ملك - رجل + امرأة ≈ ملكة`).

### 🟡 طبقة الآلية — العمليات الأساسية


In [ ]:
import numpy as np

# === 1. المتجهات والمصفوفات: البنية التحتية لكل شيء ===

v = np.array([1.0, 2.0, 3.0])          # متجه (تمثيل كلمة واحدة، بُعد 3)
M = np.array([[1, 0, 1],
              [0, 1, 1],
              [1, 1, 0]])               # مصفوفة تحويل (3×3)

# الضرب المصفوفي: جوهر كل طبقة خطية (Linear Layer) في أي شبكة عصبية
transformed_v = M @ v
print(f"المتجه بعد التحويل: {transformed_v}")

# === 2. التشابه بين المتجهات: أساس كل آلية Attention ===

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    يقيس زاوية الاتجاه بين متجهين بغض النظر عن حجمهما (magnitude).
    هذا بالضبط ما يُستخدَم لقياس "التشابه الدلالي" بين الكلمات
    في فضاء Embeddings (الفصل 7)، وكأساس نظري لعملية Attention Score
    (التي تستخدم الضرب النقطي كتقريب سريع لهذا التشابه).
    """
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b + 1e-8)

word_king = np.array([0.8, 0.6, 0.1])
word_queen = np.array([0.75, 0.65, 0.15])
word_apple = np.array([0.1, 0.1, 0.9])

print(f"تشابه (ملك، ملكة): {cosine_similarity(word_king, word_queen):.4f}")
print(f"تشابه (ملك، تفاحة): {cosine_similarity(word_king, word_apple):.4f}")
# تشابه (ملك، ملكة): 0.9959   <- قريبان دلاليًا
# تشابه (ملك، تفاحة): 0.3812  <- بعيدان دلاليًا

# === 3. القيم الذاتية والمتجهات الذاتية (Eigenvalues/Eigenvectors) ===
# مفهوم مهم لفهم PCA، وتحليل سلوك المصفوفات في الشبكات العميقة
# (مثل تفسير استقرار/عدم استقرار التدرجات عبر الطبقات)

A = np.array([[4, 1], [2, 3]])
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"\nالقيم الذاتية: {eigenvalues}")
print(f"المتجهات الذاتية:\n{eigenvectors}")


### 🟡 طبقة الآلية — الاحتمالات: softmax مستقر رقميًا


In [ ]:
def softmax_naive(logits: np.ndarray) -> np.ndarray:
    """
    التطبيق الساذج لـ softmax — يُستخدَم فقط للتوضيح التعليمي،
    لأنه غير مستقر رقميًا (قد يُسبب Overflow مع logits كبيرة).
    """
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits)

def softmax_stable(logits: np.ndarray) -> np.ndarray:
    """
    softmax المستقر رقميًا (النسخة المُستخدَمة فعليًا في كل مكتبة
    تعلّم عميق). الحيلة: طرح أقصى قيمة قبل الأس (exp) لا تُغيّر الناتج
    رياضيًا (لأن softmax(x) = softmax(x - c) لأي ثابت c) لكنها تمنع
    overflow لأن أكبر قيمة أس تُصبح exp(0)=1 بدلًا من قيمة ضخمة محتملة.
    """
    shifted_logits = logits - np.max(logits)
    exp_logits = np.exp(shifted_logits)
    return exp_logits / np.sum(exp_logits)

# توضيح المشكلة عمليًا
large_logits = np.array([1000.0, 1001.0, 1002.0])

try:
    result_naive = softmax_naive(large_logits)
    print(f"Naive softmax: {result_naive}")
except Exception as e:
    print(f"❌ Naive softmax فشل: {e}")
# ينتج: [nan, nan, nan] بسبب exp(1000) = inf ثم inf/inf = nan

result_stable = softmax_stable(large_logits)
print(f"✅ Stable softmax: {result_stable}")
# ينتج: [0.0900, 0.2447, 0.6652]  — نتيجة صحيحة ومستقرة


### 🔴 طبقة التعمّق — نظرية المعلومات: Entropy وCross-Entropy وKL Divergence


In [ ]:
def entropy(p: np.ndarray, epsilon: float = 1e-10) -> float:
    """
    الإنتروبيا (Shannon Entropy): تقيس "عدم اليقين" المتأصل في توزيع
    احتمالي. كلما كانت الإنتروبيا أعلى، كان التوزيع أكثر "انتشارًا"
    (uniform)، وكلما كانت أقل، كان أكثر "تركّزًا" حول قيمة واحدة.

        H(P) = -Σ p(x)·log(p(x))
    """
    p = np.clip(p, epsilon, 1.0)  # تجنّب log(0)
    return -np.sum(p * np.log(p))

def cross_entropy(p_true: np.ndarray, q_predicted: np.ndarray,
                   epsilon: float = 1e-10) -> float:
    """
    Cross-Entropy: تقيس "متوسط عدد البتات" اللازمة لترميز أحداث
    مسحوبة من التوزيع الحقيقي P، باستخدام كود مُصمَّم بناءً على
    التوزيع المُتوقَّع Q. هذه هي دالة الخسارة الأساسية لكل نموذج لغوي:

        H(P, Q) = -Σ p_true(x)·log(q_predicted(x))

    في التصنيف/توليد اللغة، p_true عادة one-hot (0 أو 1 فقط)،
    فتُختزَل المعادلة إلى: -log(q_predicted[الفئة_الصحيحة])
    """
    q_predicted = np.clip(q_predicted, epsilon, 1.0)
    return -np.sum(p_true * np.log(q_predicted))

def kl_divergence(p: np.ndarray, q: np.ndarray, epsilon: float = 1e-10) -> float:
    """
    KL Divergence: تقيس "المسافة" (غير المتماثلة!) بين توزيعين
    احتماليين — كم "بتة إضافية" نُهدرها باستخدام Q بدلًا من P الحقيقي.

        KL(P||Q) = Σ p(x)·log(p(x)/q(x)) = H(P,Q) - H(P)

    تُستخدَم في: قيد RLHF (الفصل 33)، Knowledge Distillation (الفصل 25)،
    وVariational Autoencoders.
    ملاحظة حرجة: KL(P||Q) ≠ KL(Q||P) — ليست مسافة بالمعنى الرياضي الدقيق!
    """
    p = np.clip(p, epsilon, 1.0)
    q = np.clip(q, epsilon, 1.0)
    return np.sum(p * np.log(p / q))

# مثال تطبيقي متكامل
true_distribution = np.array([1.0, 0.0, 0.0, 0.0])  # الكلمة الصحيحة: الفئة 0 (one-hot)

# نموذج جيد: واثق وصحيح
good_model_pred = np.array([0.85, 0.05, 0.05, 0.05])
# نموذج ضعيف: غير واثق إطلاقًا (توزيع شبه منتظم)
poor_model_pred = np.array([0.30, 0.25, 0.25, 0.20])
# نموذج سيء جدًا: واثق لكن خاطئ
wrong_confident_pred = np.array([0.05, 0.85, 0.05, 0.05])

for name, pred in [("نموذج جيد", good_model_pred),
                    ("نموذج ضعيف", poor_model_pred),
                    ("نموذج واثق وخاطئ", wrong_confident_pred)]:
    ce = cross_entropy(true_distribution, pred)
    print(f"{name}: Cross-Entropy Loss = {ce:.4f}")

# نموذج جيد: Cross-Entropy Loss = 0.1625
# نموذج ضعيف: Cross-Entropy Loss = 1.2040
# نموذج واثق وخاطئ: Cross-Entropy Loss = 2.9957  <- أسوأ عقوبة بكثير!


> 🔴 **ملاحظة جوهرية**: لاحظ كيف أن "الثقة الخاطئة" (wrong_confident_pred) تُعاقَب بشدة أكبر من "عدم اليقين" (poor_model_pred). هذا بالضبط سبب أهمية **معايرة الثقة** (Calibration) في النماذج اللغوية — سنعود لهذا المفهوم في الفصل 42 عند الحديث عن تقييم نماذج الاستدلال.

### العلاقة الرياضية الكاملة بين الدوال الثلاث


In [ ]:
def verify_relationship(p_true, q_pred):
    """
    التحقق العملي من: H(P,Q) = H(P) + KL(P||Q)
    بمعنى: Cross-Entropy = Entropy الذاتية + "الفاقد الإضافي" بسبب
    استخدام توزيع تقريبي Q بدلًا من التوزيع الحقيقي P.
    """
    h_p = entropy(p_true)
    h_pq = cross_entropy(p_true, q_pred)
    kl_pq = kl_divergence(p_true, q_pred)

    print(f"H(P) = {h_p:.4f}")
    print(f"KL(P||Q) = {kl_pq:.4f}")
    print(f"H(P) + KL(P||Q) = {h_p + kl_pq:.4f}")
    print(f"H(P,Q) = {h_pq:.4f}")
    print(f"متطابقان؟ {np.isclose(h_p + kl_pq, h_pq)}")

p = np.array([0.5, 0.3, 0.2])
q = np.array([0.4, 0.4, 0.2])
verify_relationship(p, q)
# H(P) = 1.0297
# KL(P||Q) = 0.0338
# H(P) + KL(P||Q) = 1.0635
# H(P,Q) = 1.0635
# متطابقان؟ True


---

### الصيغ المرجعية

**تشابه جيب التمام (Cosine Similarity):**

$$
\cos(\mathbf{a},\mathbf{b})
= \frac{\mathbf{a}\cdot\mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}
$$

**Softmax المستقر رقميًا:**

$$
\operatorname{softmax}(z_i)
= \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

**الإنتروبيا وCross-Entropy:**

$$
H(P) = -\sum_{x} P(x)\log P(x)
$$

$$
H(P,Q) = -\sum_{x} P(x)\log Q(x)
$$

**تباعد KL:**

$$
D_{\mathrm{KL}}(P\|Q)
= \sum_{x} P(x)\log\frac{P(x)}{Q(x)}
= H(P,Q) - H(P)
$$

هذه الصيغ الأربع هي المفردات الرياضية التي ستتكرر في بقية الكتاب: تشابه جيب التمام في التمثيلات والانتباه، وSoftmax في مخرجات النموذج، وCross-Entropy في دالة الخسارة، وKL في التقطير والمحاذاة.


## 4. مثال محلول خطوة بخطوة

**المسألة**: نموذج لغوي يُتوقَّع منه التنبؤ بالكلمة التالية من مفردات بحجم 5 كلمات {نحن, نذهب, إلى, المدرسة, اليوم}. الكلمة الصحيحة هي "المدرسة" (index=3). احسب الخسارة لثلاث تنبؤات مختلفة واشرح أيها أفضل.


In [ ]:
vocab = ["نحن", "نذهب", "إلى", "المدرسة", "اليوم"]
true_idx = 3
true_dist = np.zeros(5)
true_dist[true_idx] = 1.0

predictions = {
    "تنبؤ ممتاز": np.array([0.02, 0.02, 0.05, 0.88, 0.03]),
    "تنبؤ متوسط": np.array([0.10, 0.10, 0.20, 0.40, 0.20]),
    "تنبؤ سيء": np.array([0.30, 0.30, 0.20, 0.10, 0.10]),
}

for name, pred in predictions.items():
    loss = cross_entropy(true_dist, pred)
    perplexity = np.exp(loss)  # سنشرح Perplexity بالتفصيل في الفصل 5
    print(f"{name}: Loss={loss:.4f}, Perplexity={perplexity:.4f}, "
          f"P(المدرسة)={pred[true_idx]:.2f}")

# تنبؤ ممتاز: Loss=0.1278, Perplexity=1.1363, P(المدرسة)=0.88
# تنبؤ متوسط: Loss=0.9163, Perplexity=2.4999, P(المدرسة)=0.40
# تنبؤ سيء:  Loss=2.3026, Perplexity=10.0000, P(المدرسة)=0.10


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| استخدم دائمًا نسخة مستقرة رقميًا من softmax | `torch.softmax`/`scipy.special.softmax` تُطبِّق الحيلة تلقائيًا — لا تكتب نسخة ساذجة في كود إنتاجي أبدًا |
| احذر من `log(0)` | استخدم دائمًا `clip` أو `epsilon` صغير قبل أي عملية `log` |
| تذكّر أن KL غير متماثلة | `KL(P||Q) ≠ KL(Q||P)` — اختيار الاتجاه الصحيح مهم جدًا في RLHF وDistillation |
| استخدم `logsumexp` بدلًا من `log(sum(exp()))` | لتفادي overflow/underflow في العمليات اللوغاريتمية المركّبة (مُستخدَمة داخل PyTorch تلقائيًا في `F.cross_entropy`) |


## 6. أخطاء شائعة ومصائد

- **تطبيق softmax يدويًا دون طرح الحد الأقصى**: يُسبب `nan` مع logits كبيرة كما رأينا، وهذا خطأ شائع جدًا عند تطبيق نماذج من الصفر لأول مرة.
- **الخلط بين Entropy وCross-Entropy**: Entropy خاصية ذاتية لتوزيع واحد، بينما Cross-Entropy تقارن توزيعين (الحقيقي والمُتنبَّأ به) — هذا الخلط شائع جدًا بين المبتدئين.
- **استخدام MSE بدلًا من Cross-Entropy للتصنيف**: MSE لا يُعاقب الثقة الخاطئة بنفس الشدة المناسبة إحصائيًا لمسائل التصنيف الاحتمالي، وتدرجاتها أضعف عندما تكون الشبكة "واثقة جدًا وخاطئة" (منطقة تشبّع sigmoid/softmax).


## 7. جدول مقارنة

| الدالة | ماذا تقيس؟ | متناظرة؟ | الاستخدام الرئيسي |
|---|---|---|---|
| Entropy H(P) | عدم اليقين في توزيع واحد | — | قياس "صعوبة" توزيع، أساس Perplexity |
| Cross-Entropy H(P,Q) | تكلفة ترميز P باستخدام Q | ❌ لا | دالة الخسارة الأساسية لكل LLM |
| KL Divergence | الفرق/الفاقد الإضافي بين P وQ | ❌ لا | RLHF (KL penalty)، Distillation، VAE |
| JS Divergence | نسخة متناظرة من KL (متوسط مع توزيع وسيط) | ✅ نعم | بدائل أكثر استقرارًا أحيانًا لـ KL |


## 8. ملخص أفضل الممارسات

- افهم دائمًا الصيغة الرياضية الكاملة قبل استخدام أي دالة جاهزة من مكتبة.
- استخدم النسخ المستقرة رقميًا دائمًا (softmax, log-sum-exp) في أي تطبيق فعلي.
- راقب قيم الخسارة بعناية: قيمة `nan` مبكرة غالبًا تُشير لمشكلة استقرار عددي وليس مشكلة في البيانات بالضرورة.
- تذكّر أن معظم ما تحتاجه في LLMs هو تطبيقات مباشرة لهذه المفاهيم الثلاثة (جبر خطي + احتمالات + معلومات).


## 9. أسئلة مقابلات

**س: لماذا نستخدم Cross-Entropy Loss وليس Mean Squared Error (MSE) لتدريب نماذج التصنيف/اللغة؟**

**ج**: ثلاثة أسباب جوهرية: (1) **تفسير احتمالي صحيح**: Cross-Entropy مشتقة مباشرة من مبدأ الاستدلال الأعظمي للاحتمال (Maximum Likelihood Estimation) على توزيع Bernoulli/Categorical، بينما MSE تفترض ضمنيًا توزيعًا غاوسيًا غير مناسب للمخرجات الاحتمالية المحدودة بـ [0,1]. (2) **سلوك التدرجات**: عند استخدام softmax+MSE، تكون التدرجات صغيرة جدًا عندما يكون التنبؤ خاطئًا وواثقًا (منطقة تشبّع الدالة)، مما يُبطئ التعلّم بشكل كبير؛ بينما Cross-Entropy تُعطي تدرجًا قويًا ومتناسبًا تمامًا مع حجم الخطأ في هذه الحالة بالضبط (`∂CE/∂logit = softmax(logit) - true_label`، بسيطة ومباشرة). (3) **العقوبة المناسبة**: Cross-Entropy تُعاقب "الثقة الخاطئة" بشدة أكبر بكثير من MSE (تصل للانهاية نظريًا عند ثقة = 0 للإجابة الصحيحة)، وهذا سلوك مرغوب في مسائل التصنيف.

**س: ما العلاقة بين Perplexity وCross-Entropy؟**

**ج**: Perplexity هي ببساطة `exp(Cross-Entropy Loss)`. تفسيرها البديهي: "متوسط عدد الخيارات المتساوية الاحتمال التي يتردد النموذج بينها فعليًا عند كل تنبؤ". Perplexity=1 تعني يقينًا تامًا (مثاليًا)، بينما Perplexity=N (حجم المفردات) تعني النموذج يُخمّن عشوائيًا بالكامل. نُفضّل استخدام Perplexity كمقياس تواصلي لأنها أكثر قابلية للتفسير البديهي من قيمة Cross-Entropy الخام (لوغاريتمية).


## 10. تمرين عملي

نفّذ دالة `kl_divergence` وطبّقها لمقارنة توزيعين احتماليين مُتشابهين جدًا وآخرين مُختلفين تمامًا. ثم تحقّق تجريبيًا أن `KL(P||Q) ≠ KL(Q||P)` باستخدام نفس التوزيعين بترتيب معكوس، وفسّر دلالة الفرق العددي.


## 11. مراجع للتعمّق

- Shannon (1948) — *A Mathematical Theory of Communication*
- Goodfellow, Bengio & Courville — *Deep Learning*، الفصل 3 (Probability and Information Theory)
- Kullback & Leibler (1951) — *On Information and Sufficiency*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الأساسيات / الفصل 2: الأسس الرياضية للتعلم العميق.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
